# Week 7 - Notebook 2: Test Base Model

## Goal
Test the base Llama 3.2 model BEFORE fine-tuning to establish baseline:
1. Load base model in 4-bit quantization
2. Test on sample products
3. Measure baseline performance

Expected result: ~$110 error (terrible!)

## Time: 5-10 minutes

**Note:** This notebook requires a GPU. Run on Google Colab if you don't have one locally.

In [ ]:
import sys
sys.path.append('..')

import os
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from src.items import Item
from src.evaluator import evaluate
from src.config import config

# Load environment
load_dotenv()
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

print("✅ Environment loaded")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load Test Data

In [ ]:
print(f"Loading test data from: {config.DATASET_NAME}")
_, _, test = Item.from_hub(config.DATASET_NAME)

print(f"✅ Loaded {len(test):,} test items")
print(f"\nExample item:")
print(f"  Title: {test[0].title}")
print(f"  Price: ${test[0].price:.2f}")

## Load Base Model (4-bit Quantization)

In [ ]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {config.BASE_MODEL}")
print("This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print("✅ Model loaded in 4-bit")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## Test on Sample Products

In [ ]:
def predict_base_llama(item: Item) -> str:
    """
    Predict price using base Llama model
    """
    # Create prompt
    if not item.prompt:
        item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)
    
    prompt = item.test_prompt()
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract completion
    completion = response.split(config.PREFIX)[-1].strip()
    
    return completion

In [ ]:
# Test on a few examples
print("Testing base model on 5 sample products:\n")

for i in range(5):
    item = test[i]
    prediction = predict_base_llama(item)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: {prediction}")
    print("-" * 60)

## Full Evaluation on Test Set

In [ ]:
# Evaluate on full test set
results = evaluate(
    predict_base_llama,
    test,
    size=config.EVAL_SIZE,
    workers=1  # Sequential for GPU
)

print(f"\n{'='*60}")
print("BASE MODEL RESULTS")
print(f"{'='*60}")
print(f"Average Error: ${results['average_error']:.2f}")
print(f"MSE: {results['mse']:,.0f}")
print(f"R²: {results['r2']:.1f}%")
print(f"{'='*60}")

## Summary

✅ Base model testing complete!

**Expected results:**
- Base Llama 3.2 4-bit: ~$110 error (terrible!)
- Model has no idea how to price products
- Predictions are essentially random

**Why so bad?**
- Base model wasn't trained for price prediction
- No domain knowledge about product pricing
- Needs fine-tuning on our specific task

**Next step:** `03_finetune_colab.ipynb` - Fine-tune with QLoRA to achieve ~$40 error!